### Please run this file with Intake.txt and Patient Nutrition Needs.txt
- Intake.txt is json file loaded from patient diet recall pdf file
- Patient Nutrition Needs.txt is text from sheet DRI Progress Tracker of:
- https://docs.google.com/spreadsheets/d/1_ufMhRzRLVcmoP1_F_Q6SxrkNSDamZYM7sn2yOrB2c4/edit?gid=1382032586#gid=1382032586

#### You can manually check with food ID
https://fdc.nal.usda.gov/food-details/2452768/nutrients

In [3]:
# find food list from intake json file

import json
import pandas as pd
import requests

# Load JSON data from file
intake_file_path = "Intake.txt"  # Update with the correct path, try other Intake1.txt
with open(intake_file_path, "r", encoding="utf-8") as file:
    data = json.load(file)

# Extract food descriptions
food_items = []
for meal in data["patient"]["diet_recall"]:
    for food in meal["food"]:
        food_items.append(food["description"])

# Print the extracted food items
print("Food Items:")
print("\n".join(food_items))


Food Items:
Raisin Bran
Apple juice
Fresh peach
orange
Ground beef
Mushroom stew
Rice
Green beans
Water
Pretzels
Chocolate
Spaghetti
Ground beef
Water
Single Malt


In [5]:
# try different search method, "Foundation" provide best nutrition details, 'Survey (FNDDS)', 'SR Legacy'
payloads = [{"requireAllWords": True, "dataType": ["Foundation"]},  # excatly match
            {"requireAllWords": False, "dataType": ["Foundation"]},  # try no excatly match
            {"requireAllWords": True, "dataType": ["SR Legacy"]}, # check other type
            {"requireAllWords": True, "dataType": ["Survey (FNDDS)"]},    # check other type
            {"requireAllWords": True, "dataType": ["Branded"]},    # try "Branded" 
            {"requireAllWords": False, "dataType": ["Branded"]},    # last chance to try "Branded" without requireAllWords
          ]

API_KEY = "A2cUE0WUknfVIuJGdkebUCcKjddw1RD0bpAny1SC"
search_url = "https://api.nal.usda.gov/fdc/v1/foods/search"
headers = {"Content-Type": "application/json"}

In [6]:
# compare how similar of two strings: 
# similarity_score("apple", "Apples") => 0.91;  similarity_score("single malt", "malt") => 0.53
import difflib

def similarity_score(string1, string2):    
    similarity = difflib.SequenceMatcher(None, string1.lower(), string2.lower()).ratio()
    return round(similarity, 2)


In [7]:
# Collect nutrient data from FDC via API
nutrient_data = {}

for query in set(food_items):
    if query.lower() == 'water': # skip water here, otherwise it will search soft drinks
        continue   
    print("\n ====== query: ", query, "=========")
    similarity = 0.0
    for payload in payloads:
        # food = []
        if similarity > 0.49:
            break
        payload['query'] = query
        response = requests.post(f"{search_url}?api_key={API_KEY}", json=payload, headers=headers)
        response.raise_for_status()
        data = response.json()
        foods = data.get("foods", [])
        # print(foods)
        if len(foods) != 0:
            loops1 = 0
            for food in foods:
                
                first_word = food['description'].split(',')[0].lower()
                similarity = similarity_score(query, first_word) 
                if similarity > 0.49  or loops1 > 4:
                    print("query: ", query, "||food ID: ", food['fdcId'], "||similarity: ", similarity, 
                          "||data type: ", food['dataType'], "||food: ", food['description'])
                    
                    # print(food["foodNutrients"])
                    # get nutrition data
                    for nutrient in food["foodNutrients"]:
                        # name = nutrient["nutrient"]["nam]
                        name = nutrient["nutrientName"]
                        amount = nutrient["value"]
                        unit = nutrient["unitName"].lower()
                        label = f"{amount} {unit}" if amount is not None else "N/A"
            
                        if name not in nutrient_data:
                            nutrient_data[name] = {}
                        # nutrient_data[name][food] = label
                        nutrient_data[name][query] = label

                    break
                else:
                    print("xxx no good match(", similarity, ") ", query,":", food['description'])
                    loops1 += 1
                    print("loops: ", loops1)
        else:
            print("----- No found by", payload)

df = pd.DataFrame(nutrient_data).T
# check if a query has no result, we need create zero column for it to avoid errors afterwords
missed_food = list(set(food_items) - set(df.columns) - set({'Water'}))
if missed_food:
    print("missed those foods: ", missed_food)
    df[missed_food] = 0
df.index.name = "Nutrition"
df.to_csv("nutrition_table.csv")
df



 ====== query:  Spaghetti =========
xxx no good match( 0.43 )  Spaghetti : Sauce, pasta, spaghetti/marinara, ready-to-serve
loops:  1
xxx no good match( 0.43 )  Spaghetti : Sauce, pasta, spaghetti/marinara, ready-to-serve
loops:  1
xxx no good match( 0.12 )  Spaghetti : DENNY'S, spaghetti and meatballs
loops:  1
query:  Spaghetti ||food ID:  168912 ||similarity:  1.0 ||data type:  SR Legacy ||food:  Spaghetti, spinach, cooked

 ====== query:  Pretzels =========
----- No found by {'requireAllWords': True, 'dataType': ['Foundation'], 'query': 'Pretzels'}
----- No found by {'requireAllWords': False, 'dataType': ['Foundation'], 'query': 'Pretzels'}
xxx no good match( 0.0 )  Pretzels : Babyfood, pretzels
loops:  1
query:  Pretzels ||food ID:  169064 ||similarity:  1.0 ||data type:  SR Legacy ||food:  Pretzels, soft

 ====== query:  Single Malt =========
----- No found by {'requireAllWords': True, 'dataType': ['Foundation'], 'query': 'Single Malt'}
xxx no good match( 0.24 )  Single Malt : C

,Spaghetti,Pretzels,Single Malt,Fresh peach,orange,Chocolate,Mushroom stew,Ground beef,Green beans,Apple juice,Raisin Bran,Rice
Nutrition,,,,,,,,,,,,
Energy,130 kcal,338 kcal,0.0 kcal,174 kj,196 kj,393 kcal,NaN,NaN,NaN,NaN,NaN,NaN
Folic acid,0.0 ug,0.0 ug,NaN,NaN,NaN,0 ug,NaN,NaN,NaN,NaN,NaN,NaN
"Folate, DFE",12.0 ug,24.0 ug,NaN,NaN,NaN,3 ug,NaN,NaN,NaN,NaN,NaN,NaN
"Vitamin D (D2 + D3), International Units",0.0 iu,0.0 iu,NaN,NaN,NaN,0.0 iu,14.8 iu,NaN,NaN,NaN,NaN,NaN
Retinol,0.0 ug,0.0 ug,NaN,NaN,NaN,63 ug,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
Citric acid,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0 mg,NaN,NaN
Malic acid,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,360 mg,NaN,NaN
Oxalic acid,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0 mg,NaN,NaN


In [8]:
# seperate unit from amount, and convert unit to standard one

import pandas as pd
import re

# Load the CSV file
file_path = 'nutrition_table.csv'
df = pd.read_csv(file_path)

# Function to extract numeric value and unit
def extract_value_and_unit(cell):
    if pd.isna(cell):
        return pd.NA, pd.NA
    match = re.match(r"([\d\.]+)\s*(\w+)", str(cell))
    if match:
        return float(match.group(1)), match.group(2).lower()
    return pd.NA, pd.NA

values_df = df.copy()
units_df = pd.DataFrame(index=df.index, columns=df.columns)

for col in df.columns[1:]:
    extracted = df[col].apply(extract_value_and_unit)
    values_df[col] = extracted.apply(lambda x: x[0])
    units_df[col] = extracted.apply(lambda x: x[1])

# Define conversion factors
kj_to_kcal = 0.239005736
iu_to_ug = 0.6 # 1 IU = 0.6 mcg of beta-carotene (from food)

# Force convert "kj" to "kcal" and "iu" to "ug" regardless of other units
for col in df.columns[1:]:
    for i in values_df.index:
        unit = units_df.at[i, col]
        value = values_df.at[i, col]
        if pd.notna(unit) and pd.notna(value):
            if unit == "kj":
                values_df.at[i, col] = value * kj_to_kcal
                units_df.at[i, col] = "kcal"
            elif unit == "iu":
                values_df.at[i, col] = value * iu_to_ug
                units_df.at[i, col] = "ug"

# Determine the unit for each row
unit_column = []
for i in units_df.index:
    row_units = units_df.iloc[i, 1:].dropna().unique()
    unit_column.append(row_units[0] if len(row_units) == 1 else pd.NA)

# Append Unit column to final DataFrame
values_df["UNIT"] = unit_column
values_df.to_csv("food_nutrition_table.csv")

values_df

,Nutrition,Spaghetti,Pretzels,Single Malt,Fresh peach,orange,Chocolate,Mushroom stew,Ground beef,Green beans,Apple juice,Raisin Bran,Rice,UNIT
0,Energy,130.0,338.0,0.0,41.586998,46.845124,393.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,kcal
1,Folic acid,0.0,0.0,<NA>,<NA>,<NA>,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,ug
2,"Folate, DFE",12.0,24.0,<NA>,<NA>,<NA>,3.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,ug
3,"Vitamin D (D2 + D3), International Units",0.0,0.0,<NA>,<NA>,<NA>,0.0,8.88,<NA>,<NA>,<NA>,<NA>,<NA>,ug
4,Retinol,0.0,0.0,<NA>,<NA>,<NA>,63.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,ug
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121,Citric acid,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.0,<NA>,<NA>,mg
122,Malic acid,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,360.0,<NA>,<NA>,mg
123,Oxalic acid,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.0,<NA>,<NA>,mg
124,Quinic acid,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.0,<NA>,<NA>,mg


In [9]:
# load food intake json file again to get food intake, convert unit to ml or g

import json
import pandas as pd

# Conversion factors to grams or milliliters (approximate values)
# CONVERSIONS = {
#     "cup": (240, "ml"),       # ml
#     "oz": (29.57, "ml"),      # ml
#     "ml": (1, "ml"),          # ml
#     "g": (1, "g"),            # g
#     "medium": (150, "g")      # g
# }

CONVERSIONS = {
    # Volume-based
    "cup": (240, "ml"),
    "oz": (29.57, "ml"),
    "ml": (1, "ml"),
    "tsp": (5, "ml"),
    "tbsp": (15, "ml"),

    # Weight-based
    "g": (1, "g"),
    "kg": (1000, "g"),
    "mg": (0.001, "g"),
    "lb": (453.6, "g"),
    # "oz (weight)": (28.35, "g"),
    
    # Approximate weights for subjective portion sizes
    "medium": (150, "g"),
    "small": (100, "g"),
    "large": (200, "g"),

    # Units often used for pre-packaged or common snack items
    "bar": (40, "g"),
    "can": (355, "ml"),
    "bag": (50, "g"),
    "bottle": (500, "ml"),
    "pouch": (100, "g"),
    "slice": (100, "g"), # assume most of case we use "slice" for cake or pizza

    # "slice (bread)": (30, "g"),         # 1 slice of sandwich bread
    # "slice (cheese)": (20, "g"),        # 1 slice of processed cheese
    # "slice (ham)": (25, "g"),           # 1 slice of deli ham
    # "slice (turkey)": (25, "g"),        # 1 slice of deli turkey
    # "slice (tomato)": (20, "g"),        # 1 medium-thick tomato slice
    # "slice (apple)": (15, "g"),         # 1 apple slice
    # "slice (cake)": (80, "g"),          # 1 standard slice of cake
    # "slice (pizza)": (125, "g")         # 1 average slice of pizza

    # Liquid alcohol serving estimate
    "glass": (150, "ml"),
    "ml": (1, "ml"),
}


# Fractions to float
FRACTIONS = {
    "¼": 0.25,
    "½": 0.5,
    "¾": 0.75
}

def parse_amount(amount):
    for frac, val in FRACTIONS.items():
        amount = amount.replace(frac, str(val))
    return amount

def convert_to_float(amount):
    try:
        return eval(amount)
    except:
        return None

def convert_unit(amount_str):
    parts = amount_str.strip().split()
    if len(parts) == 2:
        value_str, unit = parts
        parsed_value_str = parse_amount(value_str)
        value = convert_to_float(parsed_value_str)
        if value is not None and unit in CONVERSIONS:
            factor, target_unit = CONVERSIONS[unit]
            return value * factor, target_unit, value
    elif len(parts) == 1 and parts[0] in CONVERSIONS:
        factor, target_unit = CONVERSIONS[parts[0]]
        return factor, target_unit, 1
    return None, None, None

# Load JSON data from file
file_path = intake_file_path #"Intake.txt"  # Update with the correct path
with open(file_path, "r", encoding="utf-8") as file:
    data = json.load(file)

# Process and convert food items
converted_foods = []
for meal in data["patient"]["diet_recall"]:
    for food in meal["food"]:
        amount_str = food["amount"]
        description = food["description"]
        converted_value, target_unit, numeric_value = convert_unit(amount_str)
        if converted_value is not None:
            rounded_value = int(converted_value) if converted_value.is_integer() else round(converted_value) # , 2)
            converted_foods.append({
                "food": description,
                "amount": amount_str,
                "amount_in_ml_or_g": rounded_value
            })
        else:
            converted_foods.append({
                "food": description,
                "amount": amount_str,
                "amount_in_ml_or_g": "unknown"
            })

# Create DataFrame
df = pd.DataFrame(converted_foods)

# Filter out unknowns
df_clean = df[df["amount_in_ml_or_g"] != "unknown"]

# Group and sum by food
df_food_summary = df_clean.groupby("food", as_index=False)["amount_in_ml_or_g"].sum()
df_food_summary.to_csv("food_summary.csv")
df_food_summary

,food,amount_in_ml_or_g
0,Apple juice,120
1,Chocolate,30
2,Fresh peach,150
3,Green beans,60
4,Ground beef,240
5,Mushroom stew,240
6,Pretzels,120
7,Raisin Bran,180
8,Rice,120
9,Single Malt,60


In [10]:
# Sum nutrition 

import pandas as pd

# Load the data
food_summary = pd.read_csv("food_summary.csv")
food_nutrition_table = pd.read_csv("food_nutrition_table.csv")

# get water amount first
water_amount = food_summary.loc[food_summary["food"] == "Water", "amount_in_ml_or_g"].values[0]

# Drop the 'Unnamed: 0' column from food_summary
food_summary = food_summary.drop(columns=["Unnamed: 0"])

# Set 'Nutrition' column as index and drop 'UNIT' temporarily
nutrition_data = food_nutrition_table.set_index("Nutrition").drop(columns=["UNIT"])

# Create a dictionary of scaling factors: {food_name: amount / 100}
# divide by 100 because nutrition table from USDA is 100ml or g
scaling_factors = dict(zip(food_summary["food"], food_summary["amount_in_ml_or_g"] / 100)) 
print(scaling_factors)
# Multiply each food column by its corresponding scaling factor (if it exists)
adjusted_nutrition = nutrition_data.copy()
for food in adjusted_nutrition.columns:
    # print('food: ', food)
    if food in scaling_factors:
        adjusted_nutrition[food] *= scaling_factors[food]
    else:
        adjusted_nutrition[food] = 0  # food not consumed that day

# Sum across all food columns to get total nutrient intake
df_nutrition_summary = adjusted_nutrition.sum(axis=1).to_frame(name="Amount")
# Extract the UNIT column from the original food_nutrition_table
unit_column = food_nutrition_table.set_index("Nutrition")["UNIT"]
# Join the unit column with df_nutrition_summary
df_nutrition_summary = df_nutrition_summary.join(unit_column)
# add water back to nutrition list, water_amount is g, covert to liter
df_nutrition_summary.loc['Water', 'Amount'] = (df_nutrition_summary.loc['Water', 'Amount'] + water_amount)/1000 
df_nutrition_summary.loc["Water", 'UNIT'] = ["liters"]

# save to csv file
# df_nutrition_summary = df_nutrition_summary.sort_index()
df_nutrition_summary.to_csv("nutrition_total_intake.csv")
df_nutrition_summary


{'Apple juice': 1.2, 'Chocolate': 0.3, 'Fresh peach': 1.5, 'Green beans': 0.6, 'Ground beef': 2.4, 'Mushroom stew': 2.4, 'Pretzels': 1.2, 'Raisin Bran': 1.8, 'Rice': 1.2, 'Single Malt': 0.6, 'Spaghetti': 2.4, 'Water': 4.74, 'orange': 1.5}


,Amount,UNIT
Nutrition,,
Energy,968.148183,kcal
Folic acid,0.000000,ug
"Folate, DFE",58.500000,ug
"Vitamin D (D2 + D3), International Units",21.312000,ug
Retinol,18.900000,ug
...,...,...
Citric acid,0.000000,mg
Malic acid,432.000000,mg
Oxalic acid,0.000000,mg


In [11]:
# Re-import required libraries due to code execution state reset
import pandas as pd
import re
import numpy as np

# Re-read the file after reset
file_path = "Patient Nutrition Needs.txt"
with open(file_path, "r") as file:
    lines = file.readlines()

# Helper function to split amount and unit properly
def parse_amount_unit(value):
    if 'low as possible' in value:
        return np.nan, ''
    if ' - ' in value:
        nums = re.findall(r"[\d.,]+", value)
        nums = [float(n.replace(',', '')) for n in nums]
        mean_val = sum(nums) / len(nums)
        unit = value.split()[-1]
        return round(mean_val, 2), unit
    if '(' in value:
        value = value.split('(')[0].strip()
    match = re.match(r"([\d.,]+)\s*([a-zA-Z/]+)", value)
    if match:
        amount, unit = match.groups()
        return float(amount.replace(',', '')), unit
    return np.nan, ''

# Reprocess the file with updated parsing
data = []
category = None
valid_categories = ['Macronutrient', 'Vitamin', 'Mineral']

for line in lines:
    line = line.strip()
    if not line:
        continue
    if any(line == cat or line.startswith(cat + '\t') for cat in valid_categories):
        category = next(cat for cat in valid_categories if line.startswith(cat))
        continue
    if line.startswith('Estimated Daily Caloric Needs'):
        value = line.split('\t')[-1]
        amount, unit = parse_amount_unit(value)
        data.append(['Calories', 'Macronutrient', amount, 'kcal'])
        continue
    parts = line.split('\t')
    if len(parts) == 2:
        nutrition = parts[0].strip()
        value = parts[1].strip()
        amount, unit = parse_amount_unit(value)
        data.append([nutrition, category, amount, unit])

# Create the cleaned DataFrame
df_final = pd.DataFrame(data, columns=['Nutrition', 'Category', 'Need_Amount', 'Need_Unit'])
df_final.to_csv("nutrition_total_needs.csv")
df_final

,Nutrition,Category,Need_Amount,Need_Unit
0,Calory,Macronutrient,3063.00,kcal
1,Carbohydrate,Macronutrient,421.50,grams
2,Total Fiber,Macronutrient,43.00,grams
3,Protein,Macronutrient,60.00,grams
4,Fat,Macronutrient,93.50,grams
5,Saturated fatty acids,Macronutrient,NaN,
6,Trans fatty acids,Macronutrient,NaN,
7,Î±-Linolenic Acid,Macronutrient,1.60,grams
8,Linoleic Acid,Macronutrient,17.00,grams
9,Dietary Cholesterol,Macronutrient,NaN,


In [12]:
import pandas as pd

# Load data
intake_df = pd.read_csv("nutrition_total_intake.csv")
needs_df = pd.read_csv("nutrition_total_needs.csv")

# Define mapping from intake_nutrition to need_nutrition
intake_to_need_mapping = {
    'Iron, Fe': 'Iron', 'Magnesium, Mg': 'Magnesium', 'Phosphorus, P': 'Phosphorus',
    'Potassium, K': 'Potassium', 'Sodium, Na': 'Sodium', 'Zinc, Zn': 'Zinc',
    'Nitrogen': None, 'Copper, Cu': 'Copper', 'Total lipid (fat)': 'Fat',
    'Thiamin': 'Thiamin', 'Manganese, Mn': 'Manganese', 'Niacin': 'Niacin',
    'Ash': None, 'Starch': None, 'Vitamin B-6': 'Vitamin B6', 'Fiber, total dietary': 'Total Fiber',
    'Biotin': 'Biotin', 'Water': 'Total Water', 'Calcium, Ca': 'Calcium',
    'Protein': 'Protein', 'Carbohydrate, by difference': 'Carbohydrate',
    'Energy (Atwater General Factors)': 'Calories', 'Energy (Atwater Specific Factors)': None,
    'Citric acid': None, 'Vitamin C, total ascorbic acid': 'Vitamin C',
    'Malic acid': None, 'Oxalic acid': None, 'Quinic acid': None, 'Folate, total': 'Folate',
    'Sucrose': None, 'Galactose': None, 'Glucose': None, 'Fructose': None,
    'Lactose': None, 'Maltose': None, 'Sugars, Total': None, 'Energy': 'Calories',
    'Cryptoxanthin, beta': None, 'Lycopene': None, 'Riboflavin': 'Riboflavin',
    'Vitamin K (Dihydrophylloquinone)': 'Vitamin K', 'Vitamin K (phylloquinone)': 'Vitamin K',
    'Vitamin A, RAE': 'Vitamin A', 'Carotene, beta': 'Carotenoids',
    'Carotene, alpha': None, 'Tryptophan': None, 'Threonine': None, 'Methionine': None,
    'Phenylalanine': None, 'Tyrosine': None, 'Alanine': None, 'Glutamic acid': None,
    'Glycine': None, 'Proline': None, 'Lutein + zeaxanthin': None,
    'Pantothenic acid': 'Pantothenic Acid', 'Selenium, Se': 'Selenium',
    'Isoleucine': None, 'Leucine': None, 'Lysine': None, 'Cystine': None,
    'Valine': None, 'Arginine': None, 'Histidine': None, 'Aspartic acid': None,
    'Serine': None, 'Fiber, insoluble': None, 'Fiber, soluble': None,
    'Carbohydrate, by summation': None, 'Cholesterol': 'Dietary Cholesterol',
    'Fatty acids, total polyunsaturated': None, 'Fatty acids, total monounsaturated': None,
    'Fatty acids, total trans': 'Trans fatty acids', 'Fatty acids, total saturated': 'Saturated fatty acids',
    'Ergothioneine': None, 'Vitamin D4': 'Vitamin D', 'Vitamin D2 (ergocalciferol)': 'Vitamin D',
    'Vitamin D (D2 + D3)': 'Vitamin D', 'Vitamin D (D2 + D3), International Units': None,
    'Delta-5-avenasterol': None, 'Ergosterol': None, 'Delta-7-Stigmastenol': None,
    'Stigmasterol': None, 'Campesterol': None, 'Beta-sitosterol': None, 'Beta-glucan': None,
    'Ergosta-7-enol': None, 'Ergosta-7,22-dienol': None, 'Ergosta-5,7-dienol': None,
    'Beta-sitostanol': None, 'Glutathione': None, 'Molybdenum, Mo': 'Molybdenum',
    'Vitamin K (Menaquinone-4)': 'Vitamin K', 'Total Sugars': None,
    'Vitamin A, IU': 'Vitamin A', 'Vitamin B-12': 'Vitamin B12'
}

# Map intake nutrients to need equivalents
intake_df["Mapped_Nutrition"] = intake_df["Nutrition"].map(intake_to_need_mapping)

# Aggregate total intake and select first unit per Mapped_Nutrition
aggregated_intake_with_unit = (
    intake_df.dropna(subset=["Mapped_Nutrition"])
    .groupby("Mapped_Nutrition")
    .agg({"Amount": "sum", "UNIT": "first"})
    .reset_index()
    .rename(columns={"Mapped_Nutrition": "Nutrition", "Amount": "Total Intake", "UNIT": "Intake_Unit"})
)

# Merge aggregated data into needs dataframe
final_needs_df = pd.merge(needs_df, aggregated_intake_with_unit, how="left", on="Nutrition").drop(columns=['Unnamed: 0'])
final_needs_df['Need_Amount'] = final_needs_df['Need_Amount'].fillna(0)
final_needs_df["Deviation"] = final_needs_df["Need_Amount"] - final_needs_df["Total Intake"]
final_needs_df.rename(columns={"Total Intake": "Intake_Amount"}, inplace=True)
final_needs_df['Intake_Amount'] = final_needs_df['Intake_Amount'].round(2)
final_needs_df['Deviation'] = final_needs_df['Deviation'].round(2)


# Save the final merged file
final_output_path = "Nutrition_Intake_vs_Needs.csv"
final_needs_df.to_csv(final_output_path, index=False)

final_needs_df


,Nutrition,Category,Need_Amount,Need_Unit,Intake_Amount,Intake_Unit,Deviation
0,Calory,Macronutrient,3063.00,kcal,2898.35,kcal,164.65
1,Carbohydrate,Macronutrient,421.50,grams,449.25,g,-27.75
2,Total Fiber,Macronutrient,43.00,grams,85.25,g,-42.25
3,Protein,Macronutrient,60.00,grams,102.32,g,-42.32
4,Fat,Macronutrient,93.50,grams,78.77,g,14.73
5,Saturated fatty acids,Macronutrient,0.00,NaN,19.45,g,-19.45
6,Trans fatty acids,Macronutrient,0.00,NaN,1.68,g,-1.68
7,Î±-Linolenic Acid,Macronutrient,1.60,grams,NaN,NaN,NaN
8,Linoleic Acid,Macronutrient,17.00,grams,NaN,NaN,NaN
9,Dietary Cholesterol,Macronutrient,0.00,NaN,168.48,mg,-168.48


In [13]:
### end 